In [2]:
import pandas as pd
import numpy as np
import os
import glob

In [3]:
customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
order_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
order_status = pd.read_csv('../data/raw/olist_orders_dataset.csv')
products = pd.read_csv('../data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')
product_category_name_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')

In [4]:
data_path="C:/Desktop/Olist-analysis/data/raw"
all_files=glob.glob(os.path.join(data_path, "*.csv"))

print(f"Found {len(all_files)} files")
all_files

Found 9 files


['C:/Desktop/Olist-analysis/data/raw\\olist_customers_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\olist_geolocation_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\olist_orders_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\olist_order_items_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\olist_order_payments_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\olist_order_reviews_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\olist_products_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\olist_sellers_dataset.csv',
 'C:/Desktop/Olist-analysis/data/raw\\product_category_name_translation.csv']

In [5]:
for f in all_files:
    df_temp = pd.read_csv(f)
    print(f"\n{os.path.basename(f)}")
    print("Shape:", df_temp.shape)
    print(df_temp.dtypes)
    print("Nulls:\n", df_temp.isnull().sum()[df_temp.isnull().sum() > 0])


olist_customers_dataset.csv
Shape: (99441, 5)
customer_id                 object
customer_unique_id          object
customer_zip_code_prefix     int64
customer_city               object
customer_state              object
dtype: object
Nulls:
 Series([], dtype: int64)

olist_geolocation_dataset.csv
Shape: (1000163, 5)
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                object
geolocation_state               object
dtype: object
Nulls:
 Series([], dtype: int64)

olist_orders_dataset.csv
Shape: (99441, 8)
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object
Nulls:
 order_approved_at                 160
order_delivere

In [6]:
order_status[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
        'order_delivered_customer_date', 'order_estimated_delivery_date']].head()

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [7]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    order_status[col] = pd.to_datetime(order_status[col], errors='coerce')

print(order_status.dtypes[date_cols])

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [8]:
print(order_status.groupby('order_status')[['order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date']].apply(lambda x: x.isna().sum()))

              order_approved_at  order_delivered_carrier_date  \
order_status                                                    
approved                      0                             2   
canceled                    141                           550   
created                       5                             5   
delivered                    14                             2   
invoiced                      0                           314   
processing                    0                           301   
shipped                       0                             0   
unavailable                   0                           609   

              order_delivered_customer_date  
order_status                                 
approved                                  2  
canceled                                619  
created                                   5  
delivered                                 8  
invoiced                                314  
processing                 

In [9]:
delivered_missing = order_status[(order_status['order_status'] == 'delivered') & (order_status['order_delivered_customer_date'].isna())]
count_delivered_missing = len(delivered_missing)
count_delivered_missing

8

In [ ]:
orders_delivered = order_status[
    (order_status['order_status'] == 'delivered') &
    (order_status['order_delivered_customer_date'].notna())
].copy()
orders_delivered.head(5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [10]:
print(order_reviews['review_score'].isna().sum())
print(order_reviews['review_score'].describe())

0
count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64


In [11]:
print(products[products['product_category_name'].isna()].shape)
missing_meta = products[products['product_category_name'].isna()]
print(missing_meta[['product_name_lenght', 'product_description_lenght', 'product_photos_qty']].isna().sum())

(610, 9)
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
dtype: int64


In [12]:
products[products['product_weight_g'].isna()]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,60.0,865.0,3.0,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
duplicate_items = order_items[order_items.duplicated(subset=['order_id', 'order_item_id'], keep=False)]
print(duplicate_items.shape)

(0, 7)


In [14]:
print(order_reviews['review_id'].duplicated().sum())
print(order_reviews['order_id'].duplicated().sum())

814
551


In [15]:
dup_reviews = order_reviews[order_reviews.duplicated(subset=['review_id'], keep=False)]
dup_reviews.sort_values('review_id').head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07 00:00:00,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21 00:00:00,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07 00:00:00,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02 00:00:00,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09 00:00:00,2017-09-13 09:52:44


In [16]:
dup_order_reviews = order_reviews[order_reviews.duplicated(subset=['order_id'], keep=False)]
dup_order_reviews.sort_values('order_id').head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25 00:00:00,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21 00:00:00,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14 00:00:00,2017-07-17 13:58:06


In [17]:
print(dup_order_reviews['order_id'].nunique())

547


In [18]:
reviews_deduped = order_reviews.sort_values('review_answer_timestamp').drop_duplicates(subset='order_id', keep='last')

print(f"Original reviews: {len(order_reviews)}")
print(f"Deduped reviews: {len(reviews_deduped)}")
print(f"Rows removed: {len(order_reviews) - len(reviews_deduped)}")

Original reviews: 99224
Deduped reviews: 98673
Rows removed: 551


In [19]:
print(products['product_id'].duplicated().sum())
print(sellers['seller_id'].duplicated().sum())
print(customers['customer_id'].duplicated().sum())
print(order_payments['order_id'].duplicated().sum())

0
0
0
4446


In [20]:
dup_payments = order_payments[order_payments.duplicated(subset='order_id', keep=False)]
dup_payments.sort_values('order_id').head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
10244,0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
16459,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
15298,009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
32058,009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45
285,009ac365164f8e06f59d18a08045f6c4,5,voucher,1,8.75


In [21]:
payment_totals = order_payments.groupby('order_id')['payment_value'].sum().reset_index()
print(payment_totals)

                               order_id  payment_value
0      00010242fe8c5a6d1ba2dd792cb16214          72.19
1      00018f77f2f0320c557190d7a144bdd3         259.83
2      000229ec398224ef6ca0657da4fc703e         216.87
3      00024acbcdf0a6daa1e931b038114c75          25.78
4      00042b26cf59d7ce69dfabb4e55b4fd9         218.04
...                                 ...            ...
99435  fffc94f6ce00a00581880bf54a75a037         343.40
99436  fffcd46ef2263f404302a634eb57f7eb         386.53
99437  fffce4705a9662cd70adb13d4a31832d         116.85
99438  fffe18544ffabc95dfada21779c9644f          64.71
99439  fffe41c64501cc87c801fd61db3f6244          55.79

[99440 rows x 2 columns]


In [22]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = 'postgres'
password = quote_plus('Ishu2005@')
host = 'localhost' 
port = '5432'
database = 'Olist'  

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

try:
    with engine.connect() as conn:
        print("Connected successfully")
except Exception as e:
    print("Connection failed:", e)

Connected successfully


In [27]:
customers.to_sql('customers', engine, if_exists='replace', index=False)
geolocation.to_sql('geolocation', engine, if_exists='replace', index=False)
order_items.to_sql('order_items', engine, if_exists='replace', index=False)
order_payments.to_sql('order_payments', engine, if_exists='replace', index=False)
reviews_deduped.to_sql('order_reviews', engine, if_exists='replace', index=False)
order_status.to_sql('order_status', engine, if_exists='replace', index=False)
orders_delivered.to_sql('orders_delivered', engine, if_exists='replace', index=False)
products.to_sql('products', engine, if_exists='replace', index=False)
sellers.to_sql('sellers', engine, if_exists='replace', index=False)
product_category_name_translation.to_sql('category_translation', engine, if_exists='replace', index=False)



71